# 동결 2,712행 남은 개선 BEFORE/SHADOW-AFTER

공개 비간섭, review-router, P1 plan-only, 32행×3회 grouped 관측을 분리 검증한다.

In [1]:
from pathlib import Path
import json
from collections import Counter
ROOT=Path.cwd()
PAIRED=ROOT/'docs/devs/LIVE_RUN_2712_REMAINING_IMPROVEMENTS_BEFORE_SHADOW_AFTER_PAIRED_20260718.jsonl'
ROUTER=ROOT/'out/regression_safe_2712_remaining_improvements_20260718/semantic_review_router_frozen2712_replay/summary.json'
PLAN=ROOT/'out/regression_safe_2712_remaining_improvements_20260718/semantic_recovery_plan_frozen2712_replay/summary.json'
GROUPED=ROOT/'out/regression_safe_2712_remaining_improvements_20260718/release_candidate_grouped_ab_v1/grouped_analysis/report.json'
paired=[json.loads(x) for x in PAIRED.read_text(encoding='utf-8').splitlines() if x.strip()]
router=json.loads(ROUTER.read_text(encoding='utf-8'))
plan=json.loads(PLAN.read_text(encoding='utf-8'))
grouped=json.loads(GROUPED.read_text(encoding='utf-8'))
len(paired), grouped['pairing']

(2712,
 {'post_id_order_sha256': '782054df2134ba35033f29ea57eea6b5a35e8934e7fcd44aa0c178cf805257d8',
  'repeat_count': 3,
  'row_count': 32,
  'status': 'EXACT'})

In [2]:
ids=[r['post_id'] for r in paired]
assert len(ids)==2712 and len(set(ids))==2712
assert all(r['noninterference']['decision_id_equal'] and r['noninterference']['stable_result_sha256_equal'] and r['noninterference']['public_projection_sha256_equal'] for r in paired)
assert all(not r['shadow_after']['router_public_apply'] and not r['shadow_after']['plan_public_apply'] for r in paired)
{'rows':len(ids),'unique':len(set(ids)),'public_drift':0,'public_apply_true':0}

{'rows': 2712, 'unique': 2712, 'public_drift': 0, 'public_apply_true': 0}

In [3]:
router_metrics=router['offline_evaluation_only']['metrics']
{k:router_metrics[k] for k in ['high_risk_union','published_intervention','fn_title_missing','fp_non_novel','wrong_work','multiwork','pass','safe_no_title','ultra_safe_no_title']}

{'high_risk_union': {'denominator': 872,
  'rate': 0.5298165137614679,
  'routed': 462},
 'published_intervention': {'denominator': 35, 'rate': 0.6, 'routed': 21},
 'fn_title_missing': {'denominator': 339,
  'rate': 0.7286135693215339,
  'routed': 247},
 'fp_non_novel': {'denominator': 72, 'rate': 0.6944444444444444, 'routed': 50},
 'wrong_work': {'denominator': 36, 'rate': 0.9166666666666666, 'routed': 33},
 'multiwork': {'denominator': 91, 'rate': 0.4945054945054945, 'routed': 45},
 'pass': {'denominator': 230, 'rate': 0.1826086956521739, 'routed': 42},
 'safe_no_title': {'denominator': 1580,
  'rate': 0.11835443037974684,
  'routed': 187},
 'ultra_safe_no_title': {'denominator': 188,
  'rate': 0.015957446808510637,
  'routed': 3}}

In [4]:
{'state_counts':plan['plan']['state_counts'], 'canonical_binding_rows':plan['plan']['canonical_binding_rows'], 'raw_ocr_rows':plan['plan']['raw_ocr_evidence_rows'], 'search_calls':plan['plan']['search_calls_executed'], 'network_calls':plan['plan']['network_calls_executed']}

{'state_counts': {'HOLD': 883, 'NOT_APPLICABLE': 1829},
 'canonical_binding_rows': 153,
 'raw_ocr_rows': 105,
 'search_calls': 0,
 'network_calls': 0}

In [5]:
q=grouped['metrics']['quality']
summary={k:{arm:q[k][arm][field] for arm in ('before','after')} for k,field in [('correct_title_recovery','majority_posts'),('false_positive','majority_posts'),('hold_integrity','majority_posts'),('pass_retention','majority_posts'),('unsafe_publish','any_run_posts'),('wrong_work_unrecovered','majority_posts')]}
summary

{'correct_title_recovery': {'before': 0, 'after': 1},
 'false_positive': {'before': 0, 'after': 0},
 'hold_integrity': {'before': 5, 'after': 5},
 'pass_retention': {'before': 1, 'after': 1},
 'unsafe_publish': {'before': 2, 'after': 3},
 'wrong_work_unrecovered': {'before': 3, 'after': 3}}

In [6]:
{'measurement_gate':grouped['measurement_gate'], 'semantic_auto_activation_gate':grouped['semantic_auto_activation_gate'], 'gate_reasons':grouped['gate_reasons'], 'full_scan_approval':False}

{'measurement_gate': 'NO_GO',
 'semantic_auto_activation_gate': 'NO_GO',
 'gate_reasons': ['SOURCE_RELATIVE_RECURRENT_REGRESSION',
  'PASS_RETENTION_INCOMPLETE',
  'WRONG_WORK_CORRECTION_UNRESOLVED',
  'MEASUREMENT_NOT_GREEN',
  'SEARCH_OR_ROUTER_NOT_GREEN'],
 'full_scan_approval': False}